# Lab 9: Build a test set and score the assistant
**Module 5.1: Measuring quality.**  **Time:** 30 minutes.

**Goal:** "it looks good in the demo" is not a quality bar. You will write test questions with expected answers,
then score the RAG assistant two ways:
* **retrieval hit:** did search find the right document?
* **answer verdict:** an LLM judge compares the answer with the expected answer (LLM-as-judge).

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
qc, _ = lk.build_policy_index()

## 1. The starter test set (5 questions)

In [ ]:
import pandas as pd
tests = lk.load_csv("data/eval_starter.csv")
pd.DataFrame(tests)

## 2. Your turn: add your own questions (10 minutes)
The cell below already has 2 extra questions. Add **at least 3 more**. For each one: copy the line marked
`TEMPLATE`, paste it on a new line above the closing `]`, delete `# TEMPLATE: ` at the start of the pasted line,
and replace the three `...` texts. Keep the comma at the end of each line.

* `question`: written the way an employee would really ask it
* `expected_answer`: the key fact only, for example `12 days`
* `source_doc`: the PDF file name that holds the answer, exactly as in the table below; for a question the
  documents cannot answer write `NOT IN DOCUMENTS` as the answer and `none` as the source

Ideas, already checked against the policies (use them, or write your own from the PDFs in `data/policies`;
in Colab: the folder icon on the left, then `course/data/policies`):

| Question | Expected answer | source_doc |
|---|---|---|
| How many sick days do I get a year? | 12 days | 01_leave_policy.pdf |
| How many unused leave days can I carry forward? | 10 days | 01_leave_policy.pdf |
| Can I fly business class to Delhi? | No, economy only, unless the CFO approves | 02_travel_expense_policy.pdf |
| By when must I submit an expense claim? | Within 30 days of the expense | 02_travel_expense_policy.pdf |
| Can I work from Goa for three weeks? | Yes, up to 4 weeks a year within India, with manager approval | 03_remote_work_policy.pdf |
| Can I paste customer data into ChatGPT? | No, restricted data must not go into public AI tools | 04_information_security_policy.pdf |
| What is the biggest gift I can accept from a supplier? | INR 2,000 per year from each partner | 09_code_of_conduct.pdf |
| What hours is phone support open? | 08:00 to 22:00 IST, seven days a week | 10_customer_support_sla.pdf |
| Who is the CEO of Meridian? | NOT IN DOCUMENTS | none |

Mix the types: plain facts, numbers, a question that needs two passages, casual wording, and at least one
question the documents cannot answer.

In [ ]:
my_tests = [
    {"question": "What is the return window for electronics?", "expected_answer": "10 days", "source_doc": "05_refund_returns_policy.pdf"},
    {"question": "What is Meridian's share price?", "expected_answer": "NOT IN DOCUMENTS", "source_doc": "none"},
    # TEMPLATE: {"question": "...", "expected_answer": "...", "source_doc": "0X_name.pdf"},
]
all_tests = tests + my_tests
print(len(all_tests), "questions")

## 3. How the judge works
Read the judge's instructions. The judge is itself an LLM, so it can be wrong. Spot-check its verdicts.

In [ ]:
print(lk.JUDGE_SYSTEM)
lk.judge("What is the minimum password length?", "14 characters", "Passwords need at least 14 characters [1].")

## 4. Score the assistant

In [ ]:
baseline = lk.evaluate(qc, all_tests, label="baseline")
baseline

## 5. Read the failures
For each FAIL decide: was it **retrieval** (wrong document found), **generation** (right document, wrong answer),
**the judge** (answer was fine), or **the test** (expected answer was unclear)?

In [ ]:
baseline[baseline.verdict == "FAIL"][["question", "retrieval_hit", "answer", "judge_reason"]]

In [ ]:
baseline.to_csv("baseline_scores.csv", index=False)   # Lab 10 compares against this file
print("saved baseline_scores.csv")

**Question:** for each FAIL in the table above, write one word: retrieval, generation, judge or test.

## Discussion (end of lab)
Who in your organisation should own the test set: the AI team, the business, or both?
How often should it grow, and from where (real user questions, incidents, complaints)?